# Chebyshev Policies and the Mountain Car Problem: Reinforcement Learning for Low-dimensional Control Tasks
## REINFORCE utilizing MLP approximators
  
As we have seen with experiments reducing the network size of PPO, performance degrades with reducing net size.  
This is why we repeat REINFORCE experiments with reasonable network sizes comparable to PPO.  
Using REINFORCE: Can MLP approximators achieve similar results as Chebyshev approximators?  

Version 1.0  
Date: 2025-10-27  
Current version: hannes.unger@fh-salzburg.ac.at  

## Imports and Definitions

In [ ]:
import copy

import time
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import multiprocessing as mp
from itertools import repeat
from utils import parallel
from pickleshare import PickleShareDB
from matplotlib import colors as mcolors
from algorithms import polynomial_agents

db = PickleShareDB('./picklesharedb')

%load_ext autoreload
%autoreload 2

In [ ]:
def get_best_candidates_of_training_result(results, optimizer_list, n_runs):
    best_index = {}
    for i in range(len(optimizer_list)):
        valid_results = []
        for j in range(n_runs):
            # Check if the element is a list and has the structure we're looking for
            if isinstance(results[i][j], list) and len(results[i][j]) > 1:
                valid_results.append((j, results[i][j][1][-1])) # take only last reward
        2
        if valid_results:  # Only proceed if there are valid results
            best_index[i] = max(valid_results, key=lambda x: x[1])[0]
        else:
            print(f'Finding index: No converging result for optimizer {i}')
            # Optionally skip optimizers with no valid results

    best_coeffs = []
    to_delete = []

    for index in best_index:
        try:
            best_coeffs.append(results[index][best_index[index]][-1])
        except:
            print(f'Adding coeffs: No converging result for optimizer {index}, removing entry')
            to_delete.append(index)
    
    for i in to_delete:
        del best_index[i]
    
    print(f'{best_index}\n{len(best_index)}\n{len(best_coeffs)}')

    return best_index, best_coeffs

In [ ]:
def get_mean_rewards_from_training_results(results, optimizer_list, n_runs, window_size=5):
    results_mean_rewards = []
    for i in range(len(optimizer_list)):
        results_mean_rewards.append(np.mean(np.array([results[i][j][1] for j in range(n_runs) if isinstance(results[i][j], list)]), axis=0)) # skip rows where nan values lead to exception instead of list
    
    # moving average for n_runs for optimizer
    results_moving_average_rewards = []

    for i in range(len(optimizer_list)):
        results_moving_average_rewards.append(np.convolve(results_mean_rewards[i], np.ones(window_size), mode='valid') / window_size)
    
    return results_mean_rewards, results_moving_average_rewards

In [ ]:
def get_mean_rewards_from_evaluation_results(results, n_runs, best_indices, window_size=5):
    results_mean_rewards = []
    to_delete = []
    for i in range(len(results)):
        try:
            results_mean_rewards.append(np.mean(np.array([results[i][j] for j in range(n_runs)]), axis=0)) 
        except:
            print(f'Calculate Mean: No converging result for optimizer {i}, removing entry')
            to_delete.append(i)

    for i in to_delete:
        del best_indices[i]

    # moving average for n_runs for optimizer
    window_size=5
    results_moving_average_rewards = []

    for i in range(len(results)):
        try:
            results_moving_average_rewards.append(np.convolve(results_mean_rewards[i], np.ones(window_size), mode='valid') / window_size)
        except:
            pass
    
    return results_mean_rewards, results_moving_average_rewards

In [ ]:
def get_converging_coeffs_of_training_results(results, optimizer_list, n_runs):
    coeffs = []

    for i, opt_result in enumerate(results):
        coeffs.append([])
        for j in range(n_runs):
            try:
                coeffs[i].append(opt_result[j][-1])
            except:
                print(f'Adding coeffs: No converging result for optimizer {i}')

    to_delete = []

    for i in range(0, len(coeffs)):
        if len(coeffs[i]) == 0:
            to_delete.append(i)

    coeffs = [val for i, val in enumerate(coeffs) if i not in to_delete]
    optimizer_list = [val for i, val in enumerate(optimizer_list) if i not in to_delete]
    return coeffs, optimizer_list

In [ ]:
def plot_evaluation_rewards(results, optimizers, window_size=5, axes=None):
    reinforce_results_eval_optimizers_mean_rewards = []
    for i in range(len(results)):
        reinforce_results_eval_optimizers_mean_rewards.append(np.mean(np.array([results[i][j] for j in range(len(results[i]))]), axis=0)) 

    # moving average for n_runs for optimizer
    window_size=5
    reinforce_results_eval_optimizers_moving_average_rewards = []

    for i in range(len(results)):
        reinforce_results_eval_optimizers_moving_average_rewards.append(np.convolve(reinforce_results_eval_optimizers_mean_rewards[i], np.ones(window_size), mode='valid') / window_size)

    if axes is None:
        fig, axes = plt.subplots(4, 3) 
        axes = axes.flatten()
        fig.set_figwidth(len(optimizers)*3)
        fig.set_figheight(20)
        fig.suptitle(f'Rewards over episodes with different optimizers')

    for i, opt in enumerate(optimizers):
        ax = axes[i]
        ax.set_title(f'{opt}')
        for j in range(len(results[i])):   
            ax.plot(results[i][j], alpha=0.3)
            ax.set_ylim([-100, 100])
        ax.plot(reinforce_results_eval_optimizers_moving_average_rewards[i], 'k', label='moving average')

In [ ]:
def plot_evaluation_min_mean_max_rewards(results, optimizers, window_size=5, color='r', label=None, axes=None, axes_indices=None, coeffs_in_results=False, ylim=True):
    if axes_indices == None:
        raise Exception('Need axis object and index information.')
    
    # mean rewards
    reinforce_results_eval_optimizers_min_rewards = []
    reinforce_results_eval_optimizers_mean_rewards = []
    reinforce_results_eval_optimizers_max_rewards = []
    for i in range(len(results)):
        if coeffs_in_results:
            reinforce_results_eval_optimizers_mean_rewards.append(np.mean(np.array([results[i][j][1] for j in range(len(results[i]))]), axis=0)) 
            reinforce_results_eval_optimizers_min_rewards.append(np.min(np.array([results[i][j][1] for j in range(len(results[i]))]), axis=0)) 
            reinforce_results_eval_optimizers_max_rewards.append(np.max(np.array([results[i][j][1] for j in range(len(results[i]))]), axis=0))     
        else:
            reinforce_results_eval_optimizers_mean_rewards.append(np.mean(np.array([results[i][j] for j in range(len(results[i]))]), axis=0)) 
            reinforce_results_eval_optimizers_min_rewards.append(np.min(np.array([results[i][j] for j in range(len(results[i]))]), axis=0)) 
            reinforce_results_eval_optimizers_max_rewards.append(np.max(np.array([results[i][j] for j in range(len(results[i]))]), axis=0)) 

    # moving average for n_runs for optimizer
    reinforce_results_eval_optimizers_moving_average_rewards = []

    for i in range(len(results)):
        reinforce_results_eval_optimizers_moving_average_rewards.append(np.convolve(reinforce_results_eval_optimizers_mean_rewards[i], np.ones(window_size), mode='valid') / window_size)

    for i, opt in enumerate(optimizers):
        ax = axes[axes_indices[opt]]
        ax.set_title(f'{opt}')        
        ax.plot(reinforce_results_eval_optimizers_max_rewards[i], color, alpha=0.5)
        ax.plot(reinforce_results_eval_optimizers_min_rewards[i], color, alpha=0.5)
        ax.plot(reinforce_results_eval_optimizers_mean_rewards[i], color, label=label)
        ax.fill_between(x=range(len(reinforce_results_eval_optimizers_max_rewards[i])), y1=reinforce_results_eval_optimizers_min_rewards[i], y2=reinforce_results_eval_optimizers_max_rewards[i], color=color, alpha=0.1)
        if ylim:
            ax.set_ylim([-100, 100])
        ax.legend(loc="best")

In [ ]:
def plot_training_rewards(results, all_optimizers, title=None, window_size=5):
    n_runs = len(results[0])
    episodes = len(results[0][0][1])

    _, reinforce_results_train_optimizers_moving_average_rewards = get_mean_rewards_from_training_results(results, all_optimizers, n_runs, window_size=5)

    fig, axes = plt.subplots(5, (len(all_optimizers)+2)//4) 
    axes = axes.flatten()
    fig.set_figwidth(len(all_optimizers)*3)
    fig.set_figheight(20)
    if not title:
        fig.suptitle(f'Rewards over episodes with different optimizers, {episodes} episodes, {n_runs} runs')
    else:
        fig.suptitle(title)

    for i, _ in enumerate(all_optimizers):
        ax = axes[i]
        ax.set_title("%s" % all_optimizers[i])
        for result in results[i]:
            try:
                ax.plot(result[1], alpha=0.3)
                ax.set_ylim([-200, 100])
            except:
                pass
        ax.plot(reinforce_results_train_optimizers_moving_average_rewards[i], 'k', label=f'mean reward (with moving average n={n_runs})')
        ax.legend(loc="best")

In [ ]:
def boxplot_training_rewards(results, all_optimizers, title=None, axes=None, label=None):
    if axes is None:
        raise Exception('Need axis object.')
    
    for i, _ in enumerate(all_optimizers):
        ax = axes[i]
        ax.set_title("%s" % all_optimizers[i])
        for result in results[i]:
            try:
                ax.boxplot(result[1], label=label)
                ax.set_ylim([-200, 100])
            except:
                pass
        ax.legend(loc="best")

In [ ]:
def plot_training_min_mean_max_rewards(results, all_optimizers, color='r', label=None, axes=None):
    for r in results:
        try:
            n_runs = len(r)
            episodes = len(r[0][1])
            break
        except:
            pass

    results_mean_rewards = []
    results_min_rewards = []
    results_max_rewards = []
    for i in range(len(all_optimizers)):
        try:
            results_mean_rewards.append(np.mean(np.array([results[i][j][1] for j in range(n_runs) if isinstance(results[i][j], list)]), axis=0)) # skip rows where nan values lead to exception instead of list
        except:
            results_mean_rewards.append(np.nan)
        try:
            results_min_rewards.append(np.min(np.array([results[i][j][1] for j in range(n_runs) if isinstance(results[i][j], list)]), axis=0)) 
        except:
            results_min_rewards.append(np.nan)
        try:    
            results_max_rewards.append(np.max(np.array([results[i][j][1] for j in range(n_runs) if isinstance(results[i][j], list)]), axis=0)) 
        except:
            results_max_rewards.append(np.nan)


    if axes is None:
        fig, axes = plt.subplots(5, (len(all_optimizers)+2)//4) 
        axes = axes.flatten()
        fig.set_figwidth(len(all_optimizers)*3)
        fig.set_figheight(20)
        fig.suptitle(f'Rewards over episodes with different optimizers, {episodes} episodes, {n_runs} runs')

    for i, _ in enumerate(all_optimizers):
        try:
            ax = axes[i]
            ax.set_title("%s" % all_optimizers[i])
            ax.plot(results_min_rewards[i], color, alpha=0.5)
            ax.plot(results_max_rewards[i], color, alpha=0.5)
            ax.plot(results_mean_rewards[i], color, label=label)
            ax.fill_between(x=range(len(results_min_rewards[i])), y1=results_min_rewards[i], y2=results_max_rewards[i], color=color, alpha=0.1)
            ax.set_ylim([-200, 100])
            ax.legend(loc="best")
        except:
            pass

In [ ]:
def get_all_eval_results_per_optimizer(results, optimizers):
    ret = {}
    for i, opt in enumerate(optimizers):
        temp_list = []
        for j in range(len(results[i])):
            try:
                temp_list.append(results[i][j])
            except:
                pass
        ret[opt] = np.concatenate(temp_list)
    return ret

In [ ]:
def get_best_candidates_of_evaluation_result(evaluation_results):
    max_mean = float('-inf')
    outer_index = -1
    inner_index = -1

    for i, outer in enumerate(evaluation_results):
        for j, inner in enumerate(outer):
            #mean_value = sum(inner) / len(inner)  # Compute mean of current inner list
            mean_value = np.mean(inner[1])
            if mean_value > max_mean:
                max_mean = mean_value
                outer_index = i
                inner_index = j

    print(f"The index with the highest mean is: {outer_index}")
    print(f"The corresponding policy index is: {inner_index}")

    return outer_index, inner_index

### Test initialization and training

In [ ]:
n_input_nodes=2
n_output_nodes=1
net_arch=[64, 64]
alpha_mu=0.0003
alpha_sigma=0.00003
episodes=100
discount=0.9 # discount=1.0 diverges for sigma approximator
initial_log_sigma=0.25 # The higher, the more initial exploration

In [ ]:
env = gym.make("MountainCarContinuous-v0", render_mode='human')
reinforce_trainable_mrp_mlp = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='mlp',
                                                                                    initial_sigma=initial_log_sigma,
                                                                                    normalize_observations=False,
                                                                                    initialization='constant',
                                                                                    mlp_n_input_nodes=n_input_nodes,
                                                                                    mlp_n_output_nodes=n_output_nodes,
                                                                                    net_arch=[64, 64])
obs = reinforce_trainable_mrp_mlp.reset()[0]
reinforce_trainable_mrp_mlp.agent.plot_me()

In [ ]:
start = time.time()
reinforce_rewards = []
reinforce_steps = []
reinforce_loss = []
_, _, _ = reinforce_trainable_mrp_mlp.train(alpha_mu=alpha_mu, alpha_sigma=alpha_sigma, epochs=episodes, discount=discount,
                                                            method='reinforce_autodiff',
                                                            learning_history=reinforce_rewards,
                                                            steps_history=reinforce_steps,
                                                            loss_history=reinforce_loss, mu_optimizer='radam', sigma_optimizer='radam')
print(f'Execution took {time.time()-start:.0f} seconds')

Execution took 68 minutes

In [ ]:
plt.plot(reinforce_rewards)

In [ ]:
# Run some actions from trained policy
obs = reinforce_trainable_mrp_mlp.reset()[0]
for i in range(600):
    reinforce_trainable_mrp_mlp.render()
    obs, reward, terminated, truncated, info, action, _ = reinforce_trainable_mrp_mlp.step(obs)  
    if terminated:
        s = reinforce_trainable_mrp_mlp.reset()

In [ ]:
mrp = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='mlp', 
                                                            normalize_observations=True,
                                                            initial_sigma=0.25,
                                                            net_arch=[64, 64],
                                                            mu_coeffs=reinforce_trainable_mrp_mlp.agent.mu_approximator.model.state_dict())      
mrp.agent.mu_approximator.model.state_dict()

In [ ]:
start = time.time()
kwargs = {'alpha_mu': alpha_mu, 'alpha_sigma': alpha_sigma, 'episodes': episodes, 'discount': discount, 'method': 'reinforce_autodiff', 'normalize_observations': True, 
          'approximator': 'mlp', 'mlp_n_input_nodes': n_input_nodes, 'mlp_n_output_nodes': n_output_nodes, 'net_arch': [64, 64]}

test_result = parallel.job_reinforce_train(kwargs)
print(f'Execution took {time.time()-start:.0f} seconds')

In [ ]:
print(f'{test_result[-1]}\n {test_result[-2]}')

### Parallel training runs

In [ ]:
n_input_nodes=2
n_output_nodes=1
net_arch = [64, 64]
alpha_mu=0.0003
alpha_sigma=0.00003
episodes=100
discount=0.9 # discount=1.0 diverges for sigma approximator
initial_sigma=0.25 # The higher, the more initial exploration

In [ ]:
kwargs = {'alpha_mu': alpha_mu, 'alpha_sigma': alpha_sigma, 'episodes': episodes, 'discount': discount, 'method': 'reinforce_autodiff', 'normalize_observations': True, 
          'approximator': 'mlp', 'mlp_n_input_nodes': n_input_nodes, 'mlp_n_output_nodes': n_output_nodes, 'net_arch': net_arch}

n_runs = 5
args = [kwargs for i in range(n_runs)]

with mp.Pool(processes=n_runs) as pool:
    reinforce_trainable_mrp_mlp_64_64_results = pool.map(parallel.job_reinforce_train, args)

# Permanently store results
db['reinforce_trainable_mrp_mlp_64_64_results'] = reinforce_trainable_mrp_mlp_64_64_results

Execution took 52 minutes.

In [ ]:
env = gym.make("MountainCarContinuous-v0", render_mode='human')
reinforce_trainable_mrp_mlp_64_64_results = db['reinforce_trainable_mrp_mlp_64_64_results']
best_index_autodiff_results = max(enumerate(reinforce_trainable_mrp_mlp_64_64_results), key=lambda x: x[1][0])[0]
reinforce_trainable_mrp_mlp_64_64 = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='mlp', normalize_observations=True, initial_sigma=initial_sigma, mu_coeffs=reinforce_trainable_mrp_mlp_64_64_results[best_index_autodiff_results][-1], sigma_coeffs=reinforce_trainable_mrp_mlp_64_64_results[best_index_autodiff_results][-2], net_arch=net_arch)
obs = reinforce_trainable_mrp_mlp_64_64.reset()[0]

In [ ]:
reinforce_trainable_mrp_mlp_64_64_results

In [ ]:
plt.plot(reinforce_trainable_mrp_mlp_64_64_results[best_index_autodiff_results][1], label='rewards')
plt.legend(loc='best')
plt.title(f'Adam: Cumulated reward over episodes, best result out of {n_runs} runs')

In [ ]:
reinforce_trainable_mrp_mlp_64_64_mean_results = np.mean([m[1] for m in reinforce_trainable_mrp_mlp_64_64_results], axis=0)
window_size=5
reinforce_trainable_mrp_mlp_64_64_moving_average_results = np.convolve(reinforce_trainable_mrp_mlp_64_64_mean_results, np.ones(window_size), mode='valid') / window_size

plt.plot(reinforce_trainable_mrp_mlp_64_64_mean_results, label='mean rewards')
plt.plot(reinforce_trainable_mrp_mlp_64_64_moving_average_results, label=f'moving average ({window_size})')
plt.legend(loc='best')
plt.title(f'Adam: Cumulated reward over episodes, mean result over {n_runs} runs\nTotal: {np.sum(reinforce_trainable_mrp_mlp_64_64_mean_results)}')

In [ ]:
# # Run some actions from trained policy
# obs = reinforce_trainable_mrp_mlp_17.reset()[0]
# for i in range(1000):
#     reinforce_trainable_mrp_mlp_17.render()
#     obs, reward, terminated, truncated, info, action, _ = reinforce_trainable_mrp_mlp_17.step(obs)  
#     if terminated:
#         s = reinforce_trainable_mrp_mlp_17.reset()

## Training and evaluation: Determine suitable learning rate

In [ ]:
# Required mlp sizes
n_input_nodes = 2
n_output_nodes = 1
net_arch = [64, 64]

episodes = 100
discount = 0.9 # discount=1.0 diverges for sigma approximator
initial_log_sigma = 0.25 # The higher, the more initial exploration
n_runs = 20

color_cheby = mcolors.CSS4_COLORS['steelblue']
color_mlp = mcolors.CSS4_COLORS['burlywood']

learning_rates = [0.3, 0.1, 0.03, 0.01, 0.003, 0.001, 0.0003, 0.0001, 0.00003, 0.00001, 0.000003, 0.000001]
kwargs = {'mode': 'optimizers', 'episodes': episodes, 'discount': discount, 'initial_sigma': initial_log_sigma, 'method': 'reinforce_autodiff', 'normalize_observations': True, 
          'n_runs': n_runs, 'approximator': 'mlp', 'net_arch': net_arch, 'mlp_n_input_nodes': n_input_nodes, 'mlp_n_output_nodes': n_output_nodes}

In [ ]:
start = time.time()
pool = parallel.NestablePool(n_runs)
reinforce_results_train_optimizers_mlp_alphas = pool.starmap(parallel.job_reinforce_learning_rate, zip(learning_rates, repeat(kwargs)))
db['reinforce_results_train_optimizers_mlp_alphas'] = reinforce_results_train_optimizers_mlp_alphas
print(f'Execution took {time.time()-start:.0f} seconds')

Execution took 130028 seconds

In [ ]:
reinforce_results_train_optimizers_mlp_alphas = db['reinforce_results_train_optimizers_mlp_alphas']
reinforce_results_train_optimizers_mean_rewards_mlp_alphas, reinforce_results_train_optimizers_moving_average_rewards_mlp_alphas = get_mean_rewards_from_training_results(reinforce_results_train_optimizers_mlp_alphas, learning_rates, n_runs)

In [ ]:
fig, axes = plt.subplots(4, (len(learning_rates)+2)//4) 
axes = axes.flatten()
fig.set_figwidth(len(learning_rates)*3)
fig.set_figheight(20)
fig.suptitle(f'Training: Min, mean and max rewards over episodes with different learning rates with net arch [64, 64], {episodes} episodes, {n_runs} runs')

plot_training_min_mean_max_rewards(reinforce_results_train_optimizers_mlp_alphas, learning_rates, color=color_mlp, axes=axes)

In [ ]:
reinforce_results_train_optimizers = db['reinforce_results_train_optimizers_mlp_alphas']
coeffs, opts = get_converging_coeffs_of_training_results(reinforce_results_train_optimizers, learning_rates, n_runs)

In [ ]:
episodes = 50
kwargs = {'episodes': episodes, 'basis': 'mlp', 'net_arch': net_arch, 'return_coeffs': True}

In [ ]:
start = time.time()
pool = parallel.NestablePool(len(coeffs))
reinforce_results_eval_optimizers_mlp_alphas = pool.starmap(parallel.job_evaluate_ncoeffs, zip(coeffs, repeat(kwargs)))
db['reinforce_results_eval_optimizers_mlp_alphas'] = reinforce_results_eval_optimizers_mlp_alphas
print(f'Execution took {time.time()-start:.0f} seconds')

In [ ]:
axes_indices = {}
for i in range(len(opts)):
    axes_indices[opts[i]] = i

reinforce_results_eval_optimizers_mlp = db['reinforce_results_eval_optimizers_mlp_alphas']

fig, axes = plt.subplots(3, 4) 
axes = axes.flatten()
fig.set_figwidth(len(opts)*3)
fig.set_figheight(20)
fig.suptitle(f'Evaluation: Moving average rewards over episodes with different learning rates and net arch [64,64] over {episodes} episodes, {n_runs} runs')

plot_evaluation_min_mean_max_rewards(reinforce_results_eval_optimizers_mlp, opts, axes=axes, axes_indices=axes_indices, color=color_mlp, ylim=False)

In [ ]:
coeffs, opt_indices = get_converging_coeffs_of_training_results(reinforce_results_train_optimizers, learning_rates, n_runs)
best_opt_index, best_opt_run = get_best_candidates_of_evaluation_result(reinforce_results_eval_optimizers_mlp)
best_opt_index, best_opt_run 

In [ ]:
env = gym.make("MountainCarContinuous-v0", render_mode='human')
reinforce_trainable_mrp_mlp_64_64 = polynomial_agents.TrainableContinuousMRPWrapper(env, basis='mlp', normalize_observations=True, initial_sigma=initial_sigma, mu_coeffs=reinforce_results_eval_optimizers_mlp[best_opt_index][best_opt_run][0], net_arch=net_arch)
obs = reinforce_trainable_mrp_mlp_64_64.reset()[0]

for i in range(1000):
    reinforce_trainable_mrp_mlp_64_64.render()
    obs, reward, terminated, truncated, info, action, _ = reinforce_trainable_mrp_mlp_64_64.step(obs)  
    if terminated:
        s = reinforce_trainable_mrp_mlp_64_64.reset()

A learning rate of 0.00001 provides the "best" results, although the policy does not manage to reach the goal flag.  

## Training and evaluation with available optimizers

We leave out optimizers that have shown not to work well in previous experiments.  

In [ ]:
# Required mlp sizes
net_arch1 = [128, 128]
net_arch2 = [64, 64]
net_arch3 = [32, 32]
net_arch4 = [16, 16]

n_input_nodes = 2
n_output_nodes = 1
alpha_mu = 0.00001
alpha_sigma = 0.00001
episodes = 100
discount = 0.9 # discount=1.0 diverges for sigma approximator
initial_sigma = 0.25 # The higher, the more initial exploration
n_runs = 20

color_cheby = mcolors.CSS4_COLORS['steelblue']
color_mlp_17 = mcolors.CSS4_COLORS['burlywood']
color_mlp_33 = mcolors.CSS4_COLORS['mediumpurple']
color_mlp_65 = mcolors.CSS4_COLORS['indianred']

optimizers = ['adam', 'adam-amsgrad', 'adamw', 'adamw-amsgrad', 'adamax', 'nadam', 'radam', 'rmsprop', 'rprop']
# kwargs_mlp_1 = {'mode': 'optimizers', 'alpha_mu': alpha_mu, 'alpha_sigma': alpha_sigma, 'episodes': episodes, 'discount': discount, 'initial_sigma': initial_sigma, 'method': 'reinforce_autodiff', 'normalize_observations': True, 'n_runs': n_runs,
#           'approximator': 'mlp', 'net_arch': net_arch1, 'mlp_n_input_nodes': n_input_nodes, 'mlp_n_output_nodes': n_output_nodes}
kwargs_mlp_2 = {'mode': 'optimizers', 'alpha_mu': alpha_mu, 'alpha_sigma': alpha_sigma, 'episodes': episodes, 'discount': discount, 'initial_sigma': initial_sigma, 'method': 'reinforce_autodiff', 'normalize_observations': True, 'n_runs': n_runs,
          'approximator': 'mlp', 'net_arch': net_arch2, 'mlp_n_input_nodes': n_input_nodes, 'mlp_n_output_nodes': n_output_nodes}
kwargs_mlp_3 = {'mode': 'optimizers', 'alpha_mu': alpha_mu, 'alpha_sigma': alpha_sigma, 'episodes': episodes, 'discount': discount, 'initial_sigma': initial_sigma, 'method': 'reinforce_autodiff', 'normalize_observations': True, 'n_runs': n_runs,
          'approximator': 'mlp', 'net_arch': net_arch3, 'mlp_n_input_nodes': n_input_nodes, 'mlp_n_output_nodes': n_output_nodes}
kwargs_mlp_4 = {'mode': 'optimizers', 'alpha_mu': alpha_mu, 'alpha_sigma': alpha_sigma, 'episodes': episodes, 'discount': discount, 'initial_sigma': initial_sigma, 'method': 'reinforce_autodiff', 'normalize_observations': True, 'n_runs': n_runs,
          'approximator': 'mlp', 'net_arch': net_arch4, 'mlp_n_input_nodes': n_input_nodes, 'mlp_n_output_nodes': n_output_nodes}

In [ ]:
# # Test function call
# #parallel.job_reinforce_optimizers(optimizers[7], kwargs)

# kwargs = {'mode': 'optimizers', 'alpha_mu': alpha_mu, 'alpha_sigma': alpha_sigma, 'episodes': episodes, 'discount': discount, 'initial_sigma': initial_sigma, 'method': 'reinforce_autodiff', 'normalize_observations': True, 'n_runs': n_runs,
#           'approximator': 'mlp', 'mlp_n_hidden_nodes': n_hidden_nodes, 'mlp_n_input_nodes': n_input_nodes, 'mlp_n_output_nodes': n_output_nodes, 'mu_optimizer': 'sgd', 'sigma_optimizer': 'sgd'}

# parallel.job_reinforce_train(kwargs=kwargs)

In [ ]:
# pool = parallel.NestablePool(n_runs)
# reinforce_results_train_optimizers_mlp_128_128 = pool.starmap(parallel.job_reinforce_optimizers, zip(optimizers, repeat(kwargs_mlp_1)))
# # Storing variables
# db['reinforce_results_train_optimizers_mlp_128_128'] = reinforce_results_train_optimizers_mlp_128_128

In [ ]:
pool = parallel.NestablePool(n_runs)
reinforce_results_train_optimizers_mlp_64_64 = pool.starmap(parallel.job_reinforce_optimizers, zip(optimizers, repeat(kwargs_mlp_2)))
# Storing variables
db['reinforce_results_train_optimizers_mlp_64_64'] = reinforce_results_train_optimizers_mlp_64_64

In [ ]:
pool = parallel.NestablePool(n_runs)
reinforce_results_train_optimizers_mlp_32_32 = pool.starmap(parallel.job_reinforce_optimizers, zip(optimizers, repeat(kwargs_mlp_3)))
# Storing variables
db['reinforce_results_train_optimizers_mlp_32_32'] = reinforce_results_train_optimizers_mlp_32_32

In [ ]:
pool = parallel.NestablePool(n_runs)
reinforce_results_train_optimizers_mlp_16_16 = pool.starmap(parallel.job_reinforce_optimizers, zip(optimizers, repeat(kwargs_mlp_4)))
# Storing variables
db['reinforce_results_train_optimizers_mlp_16_16'] = reinforce_results_train_optimizers_mlp_16_16

In [ ]:
reinforce_results_train_optimizers_mlp_64_64 = db['reinforce_results_train_optimizers_mlp_64_64']
reinforce_results_train_optimizers_mean_rewards_mlp_64_64, reinforce_results_train_optimizers_moving_average_rewards_mlp_64_64 = get_mean_rewards_from_training_results(reinforce_results_train_optimizers_mlp_64_64, optimizers, n_runs)

reinforce_results_train_optimizers_mlp_32_32 = db['reinforce_results_train_optimizers_mlp_32_32']
reinforce_results_train_optimizers_mean_rewards_mlp_32_32, reinforce_results_train_optimizers_moving_average_rewards_mlp_32_32 = get_mean_rewards_from_training_results(reinforce_results_train_optimizers_mlp_32_32, optimizers, n_runs)

reinforce_results_train_optimizers_mlp_16_16 = db['reinforce_results_train_optimizers_mlp_16_16']
reinforce_results_train_optimizers_mean_rewards_mlp_16_16, reinforce_results_train_optimizers_moving_average_rewards_mlp_16_16 = get_mean_rewards_from_training_results(reinforce_results_train_optimizers_mlp_16_16, optimizers, n_runs)

reinforce_results_train_optimizers = db['reinforce_results_train_optimizers']
reinforce_results_train_optimizers_mean_rewards, reinforce_results_train_optimizers_moving_average_rewards = get_mean_rewards_from_training_results(reinforce_results_train_optimizers, optimizers, n_runs)

reinforce_results_train_optimizers_deg5 = db['reinforce_results_train_optimizers_deg5']
reinforce_results_train_optimizers_mean_rewards_deg5, reinforce_results_train_optimizers_moving_average_rewards_deg5 = get_mean_rewards_from_training_results(reinforce_results_train_optimizers_deg5, optimizers, n_runs)

In [ ]:
fig, axes = plt.subplots(4, (len(optimizers)+2)//4) 
axes = axes.flatten()
fig.set_figwidth(len(optimizers)*3)
fig.set_figheight(20)
fig.suptitle(f'Training: Min, mean and max rewards over episodes with different optimizers, {episodes} episodes, {n_runs} runs')

plot_training_min_mean_max_rewards(reinforce_results_train_optimizers, optimizers, color=color_cheby, label='chebyshev: max-deg. 3', axes=axes)
plot_training_min_mean_max_rewards(reinforce_results_train_optimizers_deg5, optimizers, color=color_cheby, label='chebyshev: max-deg. 5', axes=axes)
plot_training_min_mean_max_rewards(reinforce_results_train_optimizers_mlp_64_64, optimizers, color=color_mlp_17, label='mlp: [64, 64]', axes=axes)
plot_training_min_mean_max_rewards(reinforce_results_train_optimizers_mlp_32_32, optimizers, color=color_mlp_33, label='mlp: [32, 32]', axes=axes)
plot_training_min_mean_max_rewards(reinforce_results_train_optimizers_mlp_16_16, optimizers, color=color_mlp_65, label='mlp: [16, 16]', axes=axes)

In [ ]:
fig, axes = plt.subplots(3, 3) 
axes = axes.flatten()
fig.set_figwidth(len(optimizers)*4)
fig.set_figheight(20)
fig.suptitle(f'Boxplots: Training rewards over episodes with different optimizers, {episodes} episodes, {n_runs} runs')

all_results = [reinforce_results_train_optimizers, reinforce_results_train_optimizers_deg5, reinforce_results_train_optimizers_mlp_16_16, reinforce_results_train_optimizers_mlp_32_32, reinforce_results_train_optimizers_mlp_64_64]

for i, _ in enumerate(optimizers):
    ax = axes[i]
    ax.set_title(optimizers[i])
    optim_boxplotdata = []
    for method in all_results:
        try:
            temp_list = []
            for j in range(len(method[i])): 
                try:
                    temp_list.append(method[i][j][1])
                except:
                    pass
            optim_boxplotdata.append(np.concatenate(temp_list))
        except:
            pass
    
    ax.boxplot(optim_boxplotdata)
    ax.set_xticks([1, 2, 3, 4, 5], ['CH-3', 'CH-5', 'mlp: [16,16]', 'mlp: [32,32]', 'mlp: [64,64]'])
    ax.set_ylim([-200, 100])

In [ ]:
fig, axes = plt.subplots(3, 3) 
axes = axes.flatten()
fig.set_figwidth(len(optimizers)*4)
fig.set_figheight(20)
fig.suptitle(f'Boxplots: Training rewards over episodes with different optimizers, {episodes} episodes, {n_runs} runs')

all_results = [reinforce_results_train_optimizers_mlp_16_16, reinforce_results_train_optimizers_mlp_32_32, reinforce_results_train_optimizers_mlp_64_64]

for i, _ in enumerate(optimizers):
    ax = axes[i]
    ax.set_title(optimizers[i])
    optim_boxplotdata = []
    for method in all_results:
        try:
            temp_list = []
            for j in range(len(method[i])): 
                try:
                    temp_list.append(method[i][j][1])
                except:
                    pass
            optim_boxplotdata.append(np.concatenate(temp_list))
        except:
            pass
    
    ax.boxplot(optim_boxplotdata)
    ax.set_xticks([1, 2, 3, 4, 5], ['CH-3', 'CH-5', 'mlp: [16,16]', 'mlp: [32,32]', 'mlp: [64,64]'])
    ax.set_ylim([-200, 100])

### Evaluation

In [ ]:
opts_long = ['adam', 'adam-amsgrad', 'adamw', 'adamw-amsgrad', 'adamax', 'lbfgs', 'nadam', 'radam', 'rmsprop', 'rprop', 'sgd-momentum', 'sgd-nesterov']
reinforce_results_train_optimizers = db['reinforce_results_train_optimizers']
coeffs, opts = get_converging_coeffs_of_training_results(reinforce_results_train_optimizers, opts_long, n_runs)

In [ ]:
reinforce_results_train_optimizers_deg5 = db['reinforce_results_train_optimizers_deg5']
coeffs_deg5, opts_deg5 = get_converging_coeffs_of_training_results(reinforce_results_train_optimizers_deg5, opts_long, n_runs)

In [ ]:
reinforce_results_train_optimizers_mlp_64_64 = db['reinforce_results_train_optimizers_mlp_64_64']
coeffs_mlp_64_64, opts_mlp_64_64 = get_converging_coeffs_of_training_results(reinforce_results_train_optimizers_mlp_64_64, optimizers, n_runs)

In [ ]:
reinforce_results_train_optimizers_mlp_32_32 = db['reinforce_results_train_optimizers_mlp_32_32']
coeffs_mlp_32_32 , opts_mlp_32_32  = get_converging_coeffs_of_training_results(reinforce_results_train_optimizers_mlp_32_32, optimizers, n_runs)

In [ ]:
reinforce_results_train_optimizers_mlp_16_16 = db['reinforce_results_train_optimizers_mlp_16_16']
coeffs_mlp_16_16, opts_mlp_16_16 = get_converging_coeffs_of_training_results(reinforce_results_train_optimizers_mlp_16_16, optimizers, n_runs)

In [ ]:
episodes = 50
kwargs_1 = {'episodes': episodes, 'basis': 'mlp', 'return_coeffs': True, 'net_arch': [64, 64]}
kwargs_2 = {'episodes': episodes, 'basis': 'mlp', 'return_coeffs': True, 'net_arch': [32, 32]}
kwargs_3 = {'episodes': episodes, 'basis': 'mlp', 'return_coeffs': True, 'net_arch': [16, 16]}

In [ ]:
pool = parallel.NestablePool(len(coeffs_mlp_64_64))
reinforce_results_eval_optimizers_mlp_64_64 = pool.starmap(parallel.job_evaluate_ncoeffs, zip(coeffs_mlp_64_64, repeat(kwargs_1)))
db['reinforce_results_eval_optimizers_mlp_64_64'] = reinforce_results_eval_optimizers_mlp_64_64

In [ ]:
pool = parallel.NestablePool(len(coeffs_mlp_32_32))
reinforce_results_eval_optimizers_mlp_32_32 = pool.starmap(parallel.job_evaluate_ncoeffs, zip(coeffs_mlp_32_32, repeat(kwargs_2)))
db['reinforce_results_eval_optimizers_mlp_32_32'] = reinforce_results_eval_optimizers_mlp_32_32

In [ ]:
pool = parallel.NestablePool(len(coeffs_mlp_16_16))
reinforce_results_eval_optimizers_mlp_16_16 = pool.starmap(parallel.job_evaluate_ncoeffs, zip(coeffs_mlp_16_16, repeat(kwargs_3)))
db['reinforce_results_eval_optimizers_mlp_16_16'] = reinforce_results_eval_optimizers_mlp_16_16

In [ ]:
axes_indices = {'adam': 0, 'adam-amsgrad': 1, 'adamw': 2, 'adamw-amsgrad': 3, 'adamax': 4, 'nadam': 5, 'radam': 6, 'rmsprop': 7, 'rprop': 8}

reinforce_results_eval_optimizers = db['reinforce_results_eval_optimizers_']
reinforce_results_eval_optimizers_deg5 = db['reinforce_results_eval_optimizers_deg5']
reinforce_results_eval_optimizers_mlp_64_64 = db['reinforce_results_eval_optimizers_mlp_64_64']
reinforce_results_eval_optimizers_mlp_32_32 = db['reinforce_results_eval_optimizers_mlp_32_32']
reinforce_results_eval_optimizers_mlp_16_16 = db['reinforce_results_eval_optimizers_mlp_16_16']

fig, axes = plt.subplots(3, 3) 
axes = axes.flatten()
fig.set_figwidth(len(optimizers)*3)
fig.set_figheight(20)
fig.suptitle(f'Evaluation: Rewards over episodes with different optimizers, {episodes} episodes, {n_runs} runs')

plot_evaluation_min_mean_max_rewards(reinforce_results_eval_optimizers, opts, axes=axes, axes_indices=axes_indices, label='chebyshev-3', color=color_cheby, coeffs_in_results=True)
plot_evaluation_min_mean_max_rewards(reinforce_results_eval_optimizers_deg5, opts, axes=axes, axes_indices=axes_indices, label='chebyshev-5', color=color_cheby)
plot_evaluation_min_mean_max_rewards(reinforce_results_eval_optimizers_mlp_16_16, opts_mlp_16_16, axes=axes, axes_indices=axes_indices, label='mlp: [16,16]', color=color_mlp_65, coeffs_in_results=True)
plot_evaluation_min_mean_max_rewards(reinforce_results_eval_optimizers_mlp_64_64, opts_mlp_64_64, axes=axes, axes_indices=axes_indices, label='mlp: [64,64]', color=color_mlp_17, coeffs_in_results=True)
plot_evaluation_min_mean_max_rewards(reinforce_results_eval_optimizers_mlp_32_32, opts_mlp_32_32, axes=axes, axes_indices=axes_indices, label='mlp: [32,32]', color=color_mlp_33, coeffs_in_results=True)

In [ ]:
def get_all_eval_results_per_optimizer(results, optimizers, coeffs_in_results=False):
    ret = {}
    for i, opt in enumerate(optimizers):
        temp_list = []
        for j in range(len(results[i])):
            try:
                if coeffs_in_results:
                    temp_list.append(results[i][j][1])
                else:
                    temp_list.append(results[i][j])
            except:
                pass
        ret[opt] = np.concatenate(temp_list)
    return ret

In [ ]:
axes_indices = {'adam': 0, 'adam-amsgrad': 1, 'adamw': 2, 'adamw-amsgrad': 3, 'adamax': 4, 'nadam': 5, 'radam': 6, 'rmsprop': 7, 'rprop': 8}

reinforce_results_eval_optimizers = db['reinforce_results_eval_optimizers']
reinforce_results_eval_optimizers_deg5 = db['reinforce_results_eval_optimizers_deg5']
reinforce_results_eval_optimizers_mlp_64_64 = db['reinforce_results_eval_optimizers_mlp_64_64']
reinforce_results_eval_optimizers_mlp_32_32 = db['reinforce_results_eval_optimizers_mlp_32_32']
reinforce_results_eval_optimizers_mlp_16_16 = db['reinforce_results_eval_optimizers_mlp_16_16']

results_chebyshev = get_all_eval_results_per_optimizer(reinforce_results_eval_optimizers, opts)
results_chebyshev_deg5 = get_all_eval_results_per_optimizer(reinforce_results_eval_optimizers_deg5, opts)
results_mlp_17 = get_all_eval_results_per_optimizer(reinforce_results_eval_optimizers_mlp_16_16, opts_mlp_16_16, coeffs_in_results=True)
results_mlp_33  = get_all_eval_results_per_optimizer(reinforce_results_eval_optimizers_mlp_32_32, opts_mlp_32_32, coeffs_in_results=True)
results_mlp_65  = get_all_eval_results_per_optimizer(reinforce_results_eval_optimizers_mlp_64_64, opts_mlp_64_64, coeffs_in_results=True)

dicts = [results_chebyshev, results_chebyshev_deg5, results_mlp_17, results_mlp_33, results_mlp_65]
all_results = {}

for d in dicts:
    for key, value in d.items():
        if key in all_results:
            all_results[key].append(value)
        else:
            all_results[key] = [value]  

fig, axes = plt.subplots(3, 3)
axes = axes.flatten()
fig.set_figwidth(len(optimizers) * 3)
fig.set_figheight(20)
fig.suptitle(f'Evaluation rewards with different optimizers, number of converging policies utilized is shown in brackets \n {episodes} episodes per policy, each datapoint is the cumulated reward of one episode')

for i, opt in enumerate(all_results):
    ax = axes[axes_indices[opt]]
    ax.set_title(opt)
    optim_boxplotdata = []
    num_policies = []
    for j, _ in enumerate(all_results[opt]):
        try:
            optim_boxplotdata.append(all_results[opt][j])
        except:
            pass
        num_policies.append(len(all_results[opt][j])/episodes)
    ax.boxplot(optim_boxplotdata)
    try:
        ax.set_xticks([1, 2, 3, 4, 5], [f'chebyshev-3 ({num_policies[0]:.0f})', f'chebyshev-5 ({num_policies[1]:.0f})', f'mlp: [16,16] ({num_policies[2]:.0f})', f'mlp: [32,32] ({num_policies[3]:.0f})', f'mlp: [64,64] ({num_policies[4]:.0f})'])
    except:
        pass
    ax.set_ylim([-200, 110])

In [ ]:
reinforce_results_train_optimizers_mlp_17 = db['reinforce_results_train_optimizers_mlp_17']
coeffs_mlp_17, opts_mlp_17 = get_converging_coeffs_of_training_results(reinforce_results_train_optimizers_mlp_17, optimizers, n_runs)

In [ ]:
reinforce_results_train_optimizers_mlp_33 = db['reinforce_results_train_optimizers_mlp_33']
coeffs_mlp_33, opts_mlp_33 = get_converging_coeffs_of_training_results(reinforce_results_train_optimizers_mlp_33, optimizers, n_runs)

In [ ]:
reinforce_results_train_optimizers_mlp_65 = db['reinforce_results_train_optimizers_mlp_65']
coeffs_mlp_65, opts_mlp_65 = get_converging_coeffs_of_training_results(reinforce_results_train_optimizers_mlp_65, optimizers, n_runs)

In [ ]:
axes_indices = {'adam': 0, 'adam-amsgrad': 1, 'adamw': 2, 'adamw-amsgrad': 3, 'adamax': 4, 'radam': 5, 'rmsprop': 6, 'rprop': 7}
titles = ['Adam', 'Adam-AMSGrad', 'AdamW', 'AdamW-AMSGrad', 'Adamax', 'NAdam', 'RAdam', 'RMSprop', 'Rprop']

reinforce_results_eval_optimizers = db['reinforce_results_eval_optimizers']
reinforce_results_eval_optimizers_deg5 = db['reinforce_results_eval_optimizers_deg5']
reinforce_results_eval_optimizers_mlp_64_64 = db['reinforce_results_eval_optimizers_mlp_64_64']
reinforce_results_eval_optimizers_mlp_32_32 = db['reinforce_results_eval_optimizers_mlp_32_32']
reinforce_results_eval_optimizers_mlp_16_16 = db['reinforce_results_eval_optimizers_mlp_16_16']
reinforce_results_eval_optimizers_mlp_4 = db['reinforce_results_eval_optimizers_mlp_17']
reinforce_results_eval_optimizers_mlp_8 = db['reinforce_results_eval_optimizers_mlp_33']
reinforce_results_eval_optimizers_mlp_16 = db['reinforce_results_eval_optimizers_mlp_65']

results_mlp_16_16 = get_all_eval_results_per_optimizer(reinforce_results_eval_optimizers_mlp_16_16, opts_mlp_16_16, coeffs_in_results=True)
results_mlp_32_32  = get_all_eval_results_per_optimizer(reinforce_results_eval_optimizers_mlp_32_32, opts_mlp_32_32, coeffs_in_results=True)
results_mlp_64_64  = get_all_eval_results_per_optimizer(reinforce_results_eval_optimizers_mlp_64_64, opts_mlp_64_64, coeffs_in_results=True)

results_mlp_17 = get_all_eval_results_per_optimizer(reinforce_results_eval_optimizers_mlp_4, opts_mlp_17, coeffs_in_results=True)
results_mlp_33  = get_all_eval_results_per_optimizer(reinforce_results_eval_optimizers_mlp_8, opts_mlp_33, coeffs_in_results=True)
results_mlp_65  = get_all_eval_results_per_optimizer(reinforce_results_eval_optimizers_mlp_16, opts_mlp_65, coeffs_in_results=True)

dicts = [results_mlp_64_64, results_mlp_32_32, results_mlp_16_16, results_mlp_65, results_mlp_33, results_mlp_17]
all_results = {}

for d in dicts:
    for key, value in d.items():
        if key in all_results:
            all_results[key].append(value)
        else:
            all_results[key] = [value]  

fig, axes = plt.subplots(2, 4)
axes = axes.flatten()
fig.subplots_adjust(hspace=0.4) 
fig.set_figwidth(32)
fig.set_figheight(12)
#fig.suptitle(f'Evaluation rewards with different optimizers, number of converging policies utilized is shown in brackets \n {episodes} episodes per policy, each datapoint is the cumulated reward of one episode')

for i, opt in enumerate(all_results):
    if opt != 'nadam':
        ax = axes[axes_indices[opt]]
        optim_boxplotdata = []
        num_policies = []
        for j, _ in enumerate(all_results[opt]):
            try:
                optim_boxplotdata.append(all_results[opt][j])
            except:
                pass
            num_policies.append(len(all_results[opt][j])/episodes)
        ax.boxplot(optim_boxplotdata)
    try:
        #ax.set_xticks([1, 2, 3, 4, 5, 6], [f'[64,64] ({num_policies[0]:.0f})', f'[32,32] ({num_policies[1]:.0f})', f'[16,16] ({num_policies[2]:.0f})', f'[16] ({num_policies[3]:.0f})', f'[8] ({num_policies[4]:.0f})', f'[4] ({num_policies[5]:.0f})' ])
        ax.set_xticks([1, 2, 3, 4, 5, 6])
        ax.set_xticklabels(
        ['[64,64]', '[32,32]', '[16,16]', '[16]', '[8]', '[4]'],
        fontsize=20,
        rotation=40,        
        ha='right',         
        rotation_mode='anchor'
        )
    except:
        pass
    ax.set_title(titles[i], fontsize=24)
    ax.set_ylim([-200, 110])
    ax.set_yticks([-200, -150, -100, -50, 0, 50, 100])
    ax.tick_params(axis='y', labelsize=20)

fig.savefig("mlp_reinforce_training_evaluation.pdf", bbox_inches='tight', pad_inches=0)